# 02 Behavior model on the processed dataset (Eating / Locomotion / Resting)

Run on a GPU runtime (T4 recommended). Dataset ZIP and review edits are in `MyDrive/CMZprojectphase2/data/`.

This notebook first creates review sheets, then trains/evaluates on the reviewed labels. Previous unreviewed results remain in `data/behavior_cv/`; this run writes to `data/behavior_cv_reviewed/`.


In [ ]:
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# CMZprojectphase2 is a friend-shared folder available in My Drive (usually via a shortcut).
DRIVE_MYDRIVE = Path("/content/drive/MyDrive")
SHARED_PROJECT_NAME = "CMZprojectphase2"
PROJECT_ROOT = DRIVE_MYDRIVE / SHARED_PROJECT_NAME
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared project folder not found: {PROJECT_ROOT}. "
        "In Google Drive, add a shortcut to the shared CMZprojectphase2 folder under My Drive, "
        "or update SHARED_PROJECT_NAME to the shortcut's name."
    )
print("Shared CMZ project:", PROJECT_ROOT)

REPO_URL = "https://github.com/Siwali/cmz-project.git"
BRANCH = "feat/civet-zoo-rules-cvat"
REPO = Path("/content/cmz-project")
PIPE = REPO / "pipeline"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout + r.stderr)
    return r.stdout + r.stderr

if REPO.exists():
    git("-C", str(REPO), "fetch", "origin", BRANCH)
    git("-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}")
else:
    git("clone", "-b", BRANCH, REPO_URL, str(REPO))
print("Using repo commit", git("-C", str(REPO), "rev-parse", "--short", "HEAD").strip())



In [ ]:
DRIVE_CMZ = PROJECT_ROOT / "data"
DATA_ZIP = DRIVE_CMZ / "cmz_processed_dataset.zip"
BASE = Path("/content/CMZ")
EDITS = DRIVE_CMZ / "review_edits.csv"
RUN_DIR = DRIVE_CMZ / "behavior_cv_reviewed"

if not (BASE / "metadata" / "manifest.csv").exists():
    if not DATA_ZIP.exists():
        raise FileNotFoundError(f"Dataset ZIP not found: {DATA_ZIP}")
    !unzip -q -n "{DATA_ZIP}" -d "{BASE}"
!ls "{BASE}" "{BASE}/metadata"



## Check the labels
Each clip's label comes from its folder name. A clip can contain a few seconds that don't match it
(for example the civet walks away at the end of a "Resting" clip). The sheets show one picture per second.

In [ ]:
!python "{PIPE}/make_review_sheets.py" --base "{BASE}" --out "{DRIVE_CMZ}/review_sheets"

In [ ]:
from IPython.display import Image, Markdown, display
review_dir = DRIVE_CMZ / "review_sheets"
for video_id in ["E05", "E02", "E03", "R05", "R06", "R07"]:
    pages = sorted(review_dir.glob(f"{video_id}_p*.jpg"))
    if pages:
        display(Markdown(f"### Review {video_id} ({len(pages)} page(s))"))
        for path in pages:
            display(Image(filename=str(path)))


Write incorrect time stretches into `MyDrive/CMZprojectphase2/data/review_edits.csv`:

```csv
video_id,start_s,end_s,action
E05,13,45,drop
E02,0,8,relabel:Locomotion
```

Use `drop` when the civet is absent or the clip label does not describe the visible behavior. Use `relabel:Eating`, `relabel:Locomotion`, or `relabel:Resting` only when the behavior is clear. Endpoints are included. Save the CSV before training.


In [ ]:
if not EDITS.exists():
    EDITS.write_text("video_id,start_s,end_s,action\n")
print(EDITS.read_text())

## Second detection pass
The dataset has saved second-pass boxes in Drive. This cell reuses them; it runs `MDV6-apa-rtdetr-e` only if the saved CSV is missing.


In [ ]:
%pip install -q PytorchWildlife

In [ ]:
# Reuse saved boxes when available; run the slow second detector only if needed.
import shutil
saved = DRIVE_CMZ / "redetect.csv"
local = BASE / "metadata" / "redetect.csv"
if saved.exists():
    shutil.copy2(saved, local)
elif local.exists():
    shutil.copy2(local, saved)
else:
    !python "{PIPE}/redetect.py" --base "{BASE}" --conf 0.15
    if local.exists():
        shutil.copy2(local, saved)
if not local.exists():
    raise FileNotFoundError(f"redetect.csv was not created: {local}")
print("redetect.csv ready:", local)


## Train and evaluate

In [ ]:
import pandas as pd
from PIL import Image, ImageDraw
from pathlib import Path

DRIVE_DATA = Path("/content/drive/MyDrive/CMZprojectphase2/data")
BASE = Path("/content/CMZ")
r = pd.read_csv(DRIVE_DATA / "redetect.csv")
print("share kept at each cut-off:", {t: round(float((r.md_conf >= t).mean()), 2) for t in (0.2, 0.3, 0.4, 0.5, 0.6)})

pick = r.sample(12, random_state=1)
sheet = Image.new("RGB", (4 * 320, 3 * 320))
for i, row in enumerate(pick.itertuples()):
    im = Image.open(BASE / row.full_path).convert("RGB")
    d = ImageDraw.Draw(im)
    d.rectangle([row.bx1, row.by1, row.bx2, row.by2], outline="red", width=4)
    d.text((10, 10), f"{row.md_conf:.2f}", fill="yellow")
    sheet.paste(im.resize((320, 320)), ((i % 4) * 320, (i // 4) * 320))
sheet

In [ ]:
!python "{PIPE}/train_behavior.py" --base "{BASE}" --edits "{EDITS}" --out "{RUN_DIR}" --cv --epochs 4 --freeze-backbone --redetect-min-conf 0.35

In [ ]:
!python "{PIPE}/stack_motion.py" --base "{BASE}" --pred "{RUN_DIR}/predictions.csv" --out "{RUN_DIR}"

In [ ]:
import json
import pandas as pd
metrics = json.loads((RUN_DIR / "metrics.json").read_text())
for k in ("model", "shortcut_baseline", "per_second"):
    v = metrics.get(k, {})
    if v:
        print(f"{k:20s} accuracy {v['accuracy']:.3f}   macro-F1 {v['macro_f1']:.3f}   rows {v['rows']}")
print("\nCNN + motion comparison:")
print(pd.read_csv(RUN_DIR / "stacked_metrics.csv").to_string(index=False))

For cross-validation, `model` and `shortcut_baseline` are scored with each video held out once. The `CNN + motion comparison` uses the same video folds. Compare those metrics with the earlier unreviewed run in `data/behavior_cv/`.


## Per-second predictions and behavior table

The next cell shows the CNN + motion per-second predictions. The behavior table below groups consecutive predictions into Active/Inactive bouts; these clips are short, so the motion script uses a 5-second minimum bout.


In [ ]:
import pandas as pd
sec = pd.read_csv(RUN_DIR / "per_second_stacked.csv")
cols = [c for c in ["video_id", "second", "label", "pred_cnn_motion", "lighting"] if c in sec.columns]
display(sec[cols].head(30))

In [ ]:
behavior_table = pd.read_csv(RUN_DIR / "behavior_table.csv")
display(behavior_table.head(30))